# E6 — Word2Vec + LSTM + Dropout

**E6 — Word2Vec (trained on our own review text) + LSTM + Dropout. Compare against E3's GloVe version.**

In [1]:
import os, re, time, json, pickle
import numpy as np
import pandas as pd

SEED = 42
DATA_PATH = "../data/IMDB Dataset.csv"
RESULTS_DIR = "../results"
TOKENIZER_PATH = "../results/tokenizer.pkl"
VOCAB_SIZE = 10000
EMBED_DIM = 100
SAMPLE_SIZE = 15000   # <-- subsample for faster training

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["review", "sentiment"])
df["review"] = df["review"].apply(lambda t: re.sub(r"<br\s*/?>", " ", str(t)))
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})
assert df["label"].isna().sum() == 0, "Unexpected sentiment values — check the column."

# Subsample BEFORE splitting, so train/test shrink together and stay balanced
df = df.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)

X = df["review"].astype(str).to_numpy()
y = df["label"].to_numpy(dtype=int)

from sklearn.model_selection import train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED,
)
print(f"Train: {len(X_train_text)}  Test: {len(X_test_text)}")

Train: 12000  Test: 3000


In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(SEED)
np.random.seed(SEED)
MAX_LEN = 200
BATCH_SIZE = 64
EPOCHS = 15
DROPOUT = 0.3
RECURRENT_DROPOUT = 0.2

I0000 00:00:1788183978.949434  103357 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788183979.510276  103357 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788183981.894876  103357 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Reuse the SAME tokenizer across every notebook (loads from disk if a
# previous notebook already built one) so all experiments share one vocab —
# that's what makes comparing accuracy across them fair.
if os.path.exists(TOKENIZER_PATH):
    with open(TOKENIZER_PATH, "rb") as f:
        tokenizer = pickle.load(f)
    print("Loaded existing tokenizer.")
else:
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_train_text)
    os.makedirs(RESULTS_DIR, exist_ok=True)
    with open(TOKENIZER_PATH, "wb") as f:
        pickle.dump(tokenizer, f)
    print("Built and saved new tokenizer.")

Loaded existing tokenizer.


In [4]:
from gensim.models import Word2Vec

def simple_tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

tokenized_train = [simple_tokenize(t) for t in X_train_text]
w2v_model = Word2Vec(sentences=tokenized_train, vector_size=EMBED_DIM, window=5,
                      min_count=2, workers=4, seed=SEED)
print(f"Word2Vec vocabulary: {len(w2v_model.wv)} words")

embedding_matrix = np.random.normal(scale=0.1, size=(VOCAB_SIZE, EMBED_DIM)).astype("float32")
hits = 0
for word, idx in tokenizer.word_index.items():
    if idx >= VOCAB_SIZE:
        continue
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]
        hits += 1
print(f"Word2Vec coverage: {hits}/{VOCAB_SIZE} ({hits/VOCAB_SIZE:.1%})")

Word2Vec vocabulary: 35304 words
Word2Vec coverage: 9798/10000 (98.0%)


In [5]:
x_train = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_LEN)
x_test = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=MAX_LEN)

model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, weights=[embedding_matrix], input_length=MAX_LEN, trainable=True),
    LSTM(64, dropout=DROPOUT, recurrent_dropout=RECURRENT_DROPOUT),
    Dropout(DROPOUT),
    Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

/home/manikya/nlp_project/venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1788183993.341111  103357 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3536 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,000,000 (3.81 MB)

 Trainable params: 1,000,000 (3.81 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
start = time.time()
history = model.fit(x_train, y_train, validation_split=0.1,
                     batch_size=BATCH_SIZE, epochs=EPOCHS, verbose=2,
                     callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)])
train_time = time.time() - start

Epoch 1/15
169/169 - 209s - 1s/step - accuracy: 0.6335 - loss: 0.6324 - val_accuracy: 0.7592 - val_loss: 0.5147
Epoch 2/15
169/169 - 185s - 1s/step - accuracy: 0.7388 - loss: 0.5254 - val_accuracy: 0.8050 - val_loss: 0.4352
Epoch 3/15
169/169 - 177s - 1s/step - accuracy: 0.7935 - loss: 0.4497 - val_accuracy: 0.8300 - val_loss: 0.4064
Epoch 4/15
169/169 - 180s - 1s/step - accuracy: 0.8330 - loss: 0.3850 - val_accuracy: 0.8450 - val_loss: 0.3773
Epoch 5/15
169/169 - 186s - 1s/step - accuracy: 0.8681 - loss: 0.3202 - val_accuracy: 0.8492 - val_loss: 0.3533
Epoch 6/15
169/169 - 186s - 1s/step - accuracy: 0.8875 - loss: 0.2815 - val_accuracy: 0.8642 - val_loss: 0.3453
Epoch 7/15
169/169 - 187s - 1s/step - accuracy: 0.9068 - loss: 0.2335 - val_accuracy: 0.8575 - val_loss: 0.3679
Epoch 8/15
169/169 - 184s - 1s/step - accuracy: 0.9221 - loss: 0.1996 - val_accuracy: 0.8667 - val_loss: 0.3668
Epoch 9/15
169/169 - 172s - 1s/step - accuracy: 0.9357 - loss: 0.1715 - val_accuracy: 0.8558 - val_loss:

In [7]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

probs = model.predict(x_test, batch_size=BATCH_SIZE).ravel()
preds = (probs > 0.5).astype(int)

acc = accuracy_score(y_test, preds)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average="binary")
auc = roc_auc_score(y_test, probs)

row = {
    "run_name": "E6_word2vec_lstm_dropout",
    "accuracy": round(acc, 4), "precision": round(prec, 4), "recall": round(rec, 4),
    "f1": round(f1, 4), "roc_auc": round(auc, 4),
    "train_time_sec": round(train_time, 1),
    "epochs_run": len(history.history["loss"]),
    "params": model.count_params(),
    "max_len": MAX_LEN, "embeddings": "word2vec", "dropout": DROPOUT,
}

os.makedirs(RESULTS_DIR, exist_ok=True)
out_path = os.path.join(RESULTS_DIR, "E6_word2vec_lstm_dropout.csv")
pd.DataFrame([row]).to_csv(out_path, index=False)
print(f"Saved {out_path}")
print(json.dumps(row, indent=2))

47/47 ━━━━━━━━━━━━━━━━━━━━ 12s 250ms/step
Saved ../results/E6_word2vec_lstm_dropout.csv
{
  "run_name": "E6_word2vec_lstm_dropout",
  "accuracy": 0.8673,
  "precision": 0.8825,
  "recall": 0.8511,
  "f1": 0.8665,
  "roc_auc": 0.9389,
  "train_time_sec": 1666.3,
  "epochs_run": 9,
  "params": 1042305,
  "max_len": 200,
  "embeddings": "word2vec",
  "dropout": 0.3
}
